# Enterprise BGE Embedding Fine-Tuning (Google Colab)

**Goal:** compare `BAAI/bge-large-en-v1.5` against a contrastively fine-tuned encoder for enterprise document retrieval. This notebook is deliberately evaluation-first: it reports Recall@5 and MRR@10 for the baseline and tuned model on the same held-out documents. Promotion is decided using validation results only; the test set is reserved for final reporting.

## Dataset decision
Do **not** repeat the previous generic FiQA-only approach: a financial QA dataset can shift the model toward financial phrasing and hurt retrieval across HR policies, technical documents, and business reports. For this optional experiment, use **synthetic enterprise retrieval triplets generated from diverse documents**. Each row needs a real question, its exact supporting passage, a plausible non-answering passage, and source document IDs. If a negative comes from another document, provide its ID so all connected source documents are assigned to one split. Include policies, annual reports, technical manuals, and business documents. Microsoft and NVIDIA PDFs alone are not enough to represent enterprise domains.

Prepare `enterprise_triplets.csv` with columns:
- `query`: realistic user question
- `positive`: exact supporting chunk (not an answer written from memory)
- `negative`: hard negative chunk that does not answer the query
- `document_id`: stable ID of the positive passage's source document
- `negative_document_id`: source ID of the negative passage; optional when the negative is from `document_id`, required when it comes from another document

Use at least a few hundred carefully checked triplets to start; more diverse high-quality examples are better than thousands of noisy synthetic examples. The split logic groups connected document IDs so neither positive nor negative source documents cross train, validation, and test. Check every positive against its source and remove unsupported or ambiguous examples.

In [ ]:
# 1) Install dependencies (Colab GPU runtime recommended)
!pip -q install -U sentence-transformers pandas scikit-learn matplotlib

In [ ]:
# 2) Upload enterprise_triplets.csv
from google.colab import files
uploaded = files.upload()
CSV_PATH = next((name for name in uploaded if name.lower().endswith('.csv')), None)
if CSV_PATH is None:
    raise ValueError("Upload a CSV with query, positive, negative, and document_id columns; negative_document_id is optional for same-document negatives")
print("Using:", CSV_PATH)

In [ ]:
# 3) Validate and split by every positive/negative source document to prevent leakage
import os, random, json, numpy as np, pandas as pd, torch
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
df = pd.read_csv(CSV_PATH).fillna("")
required = {"query", "positive", "negative", "document_id"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
for col in required:
    df[col] = df[col].astype(str).str.strip()
if "negative_document_id" not in df.columns:
    df["negative_document_id"] = df["document_id"]
else:
    df["negative_document_id"] = df["negative_document_id"].astype(str).str.strip()
    df.loc[df["negative_document_id"] == "", "negative_document_id"] = df["document_id"]
df = df[(df["query"] != "") & (df["positive"] != "") & (df["negative"] != "") & (df["document_id"] != "")].copy()
df = df.drop_duplicates(subset=["query", "positive", "negative", "document_id", "negative_document_id"])
if len(df) < 50:
    print("WARNING: fewer than 50 triplets; results will be highly uncertain.")
# Merge documents connected by positive/negative pairs. The same source document
# must remain in one split even when it appears as a negative elsewhere.
parent = {}
def find(document_id):
    parent.setdefault(document_id, document_id)
    if parent[document_id] != document_id:
        parent[document_id] = find(parent[document_id])
    return parent[document_id]
def union(left, right):
    left_root, right_root = find(left), find(right)
    if left_root != right_root:
        parent[right_root] = left_root
for positive_id, negative_id in zip(df["document_id"], df["negative_document_id"]):
    union(positive_id, negative_id)
df["document_group"] = [find(document_id) for document_id in df["document_id"]]
if df["document_group"].nunique() < 10:
    raise ValueError("Need at least 10 independent document groups after grouping positive/negative sources.")
# Approximate 80/10/10 split by connected source-document group.
split1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, hold_idx = next(split1.split(df, groups=df["document_group"]))
train_df, hold_df = df.iloc[train_idx].copy(), df.iloc[hold_idx].copy()
split2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED+1)
val_rel, test_rel = next(split2.split(hold_df, groups=hold_df["document_group"]))
val_df, test_df = hold_df.iloc[val_rel].copy(), hold_df.iloc[test_rel].copy()
# If group split leaves a split empty (few documents), stop rather than silently leak.
if min(len(train_df), len(val_df), len(test_df)) == 0:
    raise ValueError("Group split produced an empty split. Add more distinct source documents.")
def source_documents(frame):
    return set(frame["document_id"]) | set(frame["negative_document_id"])
assert source_documents(train_df).isdisjoint(source_documents(hold_df))
assert source_documents(val_df).isdisjoint(source_documents(test_df))
print({"rows":len(df),"train":len(train_df),"validation":len(val_df),"test":len(test_df),"source_documents":len(source_documents(df)),"independent_groups":df["document_group"].nunique(),"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})

### Dataset quality gate
Before training, manually inspect a sample of triplets. A hard negative must be topically plausible but not answer the query. Random unrelated text makes the task too easy and often gives misleadingly high metrics. If Microsoft/NVIDIA are the only two documents currently available, **do not train on them and report the result as general enterprise improvement**; first add more source documents, or use them only for a small pilot.

In [ ]:
# 4) Load BGE baseline and define chunk-level retrieval metrics
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sentence_transformers.evaluation import SequentialEvaluator

BASE_MODEL = "BAAI/bge-large-en-v1.5"
MAX_SEQ_LENGTH = 384
model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = MAX_SEQ_LENGTH

def retrieval_metrics(encoder, frame, k=5, mrr_k=10):
    """Candidate pool is all held-out positives and hard negatives for held-out queries."""
    if frame.empty:
        raise ValueError("Evaluation split is empty")
    candidates = list(dict.fromkeys(frame.positive.tolist() + frame.negative.tolist()))
    cand_emb = encoder.encode(candidates, normalize_embeddings=True, batch_size=32, convert_to_numpy=True, show_progress_bar=False)
    query_emb = encoder.encode(frame.query.tolist(), normalize_embeddings=True, batch_size=16, convert_to_numpy=True, show_progress_bar=False)
    cand_index = {text:i for i,text in enumerate(candidates)}
    recalls, reciprocal_ranks = [], []
    for qemb, row in zip(query_emb, frame.itertuples(index=False)):
        scores = cand_emb @ qemb
        order = np.argsort(-scores)
        positive_idx = cand_index[row.positive]
        rank = int(np.where(order == positive_idx)[0][0]) + 1
        # Each query has one annotated positive. Recall@5 is 1 iff that positive is in top 5.
        recalls.append(float(rank <= k))
        reciprocal_ranks.append(1.0 / rank if rank <= mrr_k else 0.0)
    return {f"Recall@{k}": float(np.mean(recalls)), f"MRR@{mrr_k}": float(np.mean(reciprocal_ranks)), "queries": len(frame)}

baseline_val = retrieval_metrics(model, val_df)
baseline_test = retrieval_metrics(model, test_df)
print("BASELINE validation:", baseline_val)
print("BASELINE test:", baseline_test)

In [ ]:
# 5) Train with triplet contrastive learning
# Colab T4: batch size 4 is conservative for BGE-large. Reduce to 2 if CUDA OOM.
BATCH_SIZE = 4
EPOCHS = 1                 # start with 1; use validation results before increasing
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
OUTPUT_DIR = "/content/bge-enterprise-finetuned"

train_examples = [InputExample(texts=[r.query, r.positive, r.negative]) for r in train_df.itertuples(index=False)]
train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE, drop_last=True)
train_loss = losses.TripletLoss(model=model, distance_metric=losses.TripletDistanceMetric.COSINE, triplet_margin=0.2)
steps_per_epoch = max(1, len(train_loader))
warmup_steps = int(steps_per_epoch * EPOCHS * WARMUP_RATIO)
model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LEARNING_RATE},
    output_path=OUTPUT_DIR,
    show_progress_bar=True,
    use_amp=torch.cuda.is_available(),
)
print("Saved candidate model to", OUTPUT_DIR)

In [ ]:
# 6) Compare baseline and fine-tuned model on identical held-out data
fine_tuned = SentenceTransformer(OUTPUT_DIR)
fine_tuned.max_seq_length = MAX_SEQ_LENGTH
results = {
    "baseline_validation": baseline_val,
    "finetuned_validation": retrieval_metrics(fine_tuned, val_df),
    "baseline_test": baseline_test,
    "finetuned_test": retrieval_metrics(fine_tuned, test_df),
}
print(json.dumps(results, indent=2))
with open("embedding_evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)
files.download("embedding_evaluation_results.json")

In [ ]:
# 7) Make the promotion decision from validation only; test is report-only.
val_b = results["baseline_validation"]; val_f = results["finetuned_validation"]
promote = (val_f["Recall@5"] >= val_b["Recall@5"] and val_f["MRR@10"] >= val_b["MRR@10"]
           and (val_f["Recall@5"] > val_b["Recall@5"] or val_f["MRR@10"] > val_b["MRR@10"]))
print("Validation-based promotion decision:", promote)
if promote:
    import shutil
    shutil.make_archive("bge-enterprise-finetuned", "zip", OUTPUT_DIR)
    files.download("bge-enterprise-finetuned.zip")
else:
    print("Do not replace baseline. Improve triplet quality/domain diversity or try a lower learning rate / one more epoch, then rerun.")

## Integration notes
- BGE-large outputs **1024-dimensional** vectors. Confirm the exported model keeps the same dimension before using it in the existing pgvector column.
- Set `EMBEDDING_MODEL_NAME` to the extracted model directory and keep `EMBEDDING_DIMENSION=1024` only after verifying the dimension.
- **Re-embed every stored chunk** after switching encoders. Do not mix vectors from baseline and fine-tuned encoders in the same index.
- The test metrics here are a controlled triplet-candidate evaluation. Before making a strong claim, also evaluate on a fixed, manually checked query-to-chunk benchmark with realistic distractor chunks, including unanswerable questions.
- If the fine-tuned model loses to baseline, keep baseline. A lower training loss is not proof of better retrieval.